# Results paragraph: Longitudinal harmonization and vendor effects

Uses assembled vendor-effect outputs (ΔR²adj = full model minus reduced model without scanner vendor) to fill the longitudinal harmonization results paragraph. Data: `data/vendor_effects/vendor_effects_all_outputs.rds`. Filter: `qc_covariate == "no_quality"`, five metrics (MKT, ICVF, RTOP, FA, MD).

**Figure 3:** Panel A = raw vendor-effect heatmap; Panels B/C = mean vendor effects before vs after harmonization.

## Setup and load vendor effects

In [1]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(fs)
  library(jsonlite)
})

config_candidates <- c(
  Sys.getenv("CONFIG_PATH", unset = ""),
  fs::path(".", "config.json"),
  fs::path("..", "config.json"),
  fs::path("..", "..", "config.json")
)
config_candidates <- normalizePath(unique(config_candidates[nzchar(config_candidates)]), winslash = "/", mustWork = FALSE)
config_path <- config_candidates[file.exists(config_candidates)][1]
if (is.na(config_path) || !nzchar(config_path)) stop("Could not locate config.json.")
config <- jsonlite::fromJSON(config_path)
project_root <- normalizePath(config$project_root, winslash = "/", mustWork = FALSE)

vendor_rds <- fs::path(project_root, "data", "vendor_effects", "vendor_effects_all_outputs.rds")
if (!file.exists(vendor_rds)) stop("Vendor effects file not found: ", vendor_rds)

df_vendor <- readRDS(vendor_rds)
if (!is.data.frame(df_vendor)) stop("Vendor effects file is not a data.frame.")

qc_target <- "no_quality"
metrics_keep <- c("DKI_mkt", "NODDI_icvf", "MAPMRI_rtop", "GQI_fa", "GQI_md")
metric_labels <- c(
  DKI_mkt = "MKT",
  NODDI_icvf = "ICVF",
  MAPMRI_rtop = "RTOP",
  GQI_fa = "FA",
  GQI_md = "MD"
)

df <- df_vendor %>%
  filter(metric %in% metrics_keep, !is.na(effect_size))
if ("qc_covariate" %in% names(df) && qc_target %in% df$qc_covariate) {
  df <- df %>% filter(qc_covariate == qc_target)
}
cat("N rows (no_quality, five metrics):", nrow(df), "\n")

N rows (no_quality, five metrics): 620 


## Extract numbers for paragraph

In [2]:
# Raw data only (univariate level, Fig 3a)
raw <- df %>% filter(source == "raw")
if (nrow(raw) == 0) stop("No raw vendor effects found.")

# Range of vendor effects (raw): "These effects ranged between X-X" (as %)
effect_pct_min <- min(raw$effect_size, na.rm = TRUE) * 100
effect_pct_max <- max(raw$effect_size, na.rm = TRUE) * 100
range_lo <- round(effect_pct_min, 1)
range_hi <- round(effect_pct_max, 1)

# Average vendor effect (%) by metric (raw), for MKT, RTOP, MD
raw_by_metric <- raw %>%
  group_by(metric) %>%
  summarise(mean_effect_pct = mean(effect_size, na.rm = TRUE) * 100, .groups = "drop")

mkt_avg_pct <- raw_by_metric %>% filter(metric == "DKI_mkt") %>% pull(mean_effect_pct) %>% `[`(1)
rtop_avg_pct <- raw_by_metric %>% filter(metric == "MAPMRI_rtop") %>% pull(mean_effect_pct) %>% `[`(1)
md_avg_pct <- raw_by_metric %>% filter(metric == "GQI_md") %>% pull(mean_effect_pct) %>% `[`(1)
mkt_avg_pct <- round(replace(mkt_avg_pct, is.na(mkt_avg_pct), 0), 1)
rtop_avg_pct <- round(replace(rtop_avg_pct, is.na(rtop_avg_pct), 0), 1)
md_avg_pct <- round(replace(md_avg_pct, is.na(md_avg_pct), 0), 1)

# Harmonized: average effect (%) by metric; paragraph says "reduced to <X% for all"
harm <- df %>% filter(source == "harmonized")
harm_by_metric <- harm %>%
  group_by(metric) %>%
  summarise(mean_effect_pct = mean(effect_size, na.rm = TRUE) * 100, .groups = "drop")

max_harm_avg <- if (nrow(harm_by_metric) > 0) max(harm_by_metric$mean_effect_pct, na.rm = TRUE) else 0
cap_pct <- ceiling(max_harm_avg)
if (!is.finite(cap_pct) || cap_pct < 1) cap_pct <- 1

cat("Raw effect range (%):", range_lo, "-", range_hi, "\n")
cat("Raw MKT avg %:", mkt_avg_pct, "\n")
cat("Raw RTOP avg %:", rtop_avg_pct, "\n")
cat("Raw MD avg %:", md_avg_pct, "\n")
cat("Harmonized max mean effect (%):", round(max_harm_avg, 2), "=> paragraph <", cap_pct, "%\n")

# Min, max, and average vendor effect (%%) across bundles within each metric (raw and harmonized)
effect_summary <- df %>%
  mutate(effect_pct = effect_size * 100) %>%
  group_by(source, metric) %>%
  summarise(
    min_pct = min(effect_pct, na.rm = TRUE),
    max_pct = max(effect_pct, na.rm = TRUE),
    mean_pct = mean(effect_pct, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(metric_label = unname(metric_labels[metric])) %>%
  mutate(across(c(min_pct, max_pct, mean_pct), ~ round(., 2)))

cat("\nVendor effect (%% variance) across bundles by metric:\n")
print(effect_summary)

Raw effect range (%): -0.1 - 66.3 
Raw MKT avg %: 28.2 
Raw RTOP avg %: 2.8 
Raw MD avg %: 9.4 
Harmonized max mean effect (%): -0.01 => paragraph < 1 %


## Filled paragraph

In [3]:
txt <- sprintf(
  paste(
    "Although harmonization has previously been applied to ABCD dMRI data, prior efforts have focused on cross-sectional baseline scans.",
    "Here, we implemented an advanced longitudinal harmonization approach that preserves both participant-specific effects as well as nonlinear developmental trajectories.",
    "We first tested whether longitudinal harmonization reduced mean vendor-related variance in ABCD.",
    "First, we modeled each metric in each bundle as a function of age, sex, and scanner vendor. The vendor effect size was operationalized as the delta-R2adj between the full model and a reduced model excluding the scanner vendor term. This process was performed on the raw and harmonized data separately.",
    "At the univariate level, scanner vendor accounted for substantial variance in several microstructural measures prior to harmonization. These effects ranged between %s-%s%%, and were not uniform across bundles and microstructural measures (Fig. 3a).",
    "For example, in the raw data, scanner vendor explained %s%% of the variance in MKT, averaged across bundles (Fig. 3b,c). In contrast, both RTOP (average effect of %s%% across bundles) and MD (average effect of %s%% across bundles) had comparatively smaller susceptibility to scanner vendor.",
    "Following harmonization, average vendor-related effects across bundles were reduced to <%s%%%% for all FA, MD, RTOP, ICVF, and MKT.",
    "These results indicate that longitudinal nonlinear harmonization is feasible on ABCD and markedly attenuates vendor-driven differences in microstructural values.",
    sep = " "
  ),
  range_lo, range_hi,
  mkt_avg_pct,
  rtop_avg_pct, md_avg_pct,
  cap_pct
)
cat(txt, "\n")

Although harmonization has previously been applied to ABCD dMRI data, prior efforts have focused on cross-sectional baseline scans. Here, we implemented an advanced longitudinal harmonization approach that preserves both participant-specific effects as well as nonlinear developmental trajectories. We first tested whether longitudinal harmonization reduced mean vendor-related variance in ABCD. First, we modeled each metric in each bundle as a function of age, sex, and scanner vendor. The vendor effect size was operationalized as the delta-R2adj between the full model and a reduced model excluding the scanner vendor term. This process was performed on the raw and harmonized data separately. At the univariate level, scanner vendor accounted for substantial variance in several microstructural measures prior to harmonization. These effects ranged between -0.1-66.3%, and were not uniform across bundles and microstructural measures (Fig. 3a). For example, in the raw data, scanner vendor expla